# 01 - Preparação de Dados

Este notebook carrega o arquivo `espelhos_acordaos_artigo2026.parquet` e complementa cada registro com o texto do espelho obtido a partir do `id_espelho` na **Base de Dados Abertos do STJ** (CKAN).

## Fluxo
1. Carrega o parquet de espelhos base.
2. Mapeia os recursos disponíveis em cada dataset do CKAN.
3. Para cada `id_espelho`, consulta o datastore do CKAN e recupera o texto.
4. Salva o resultado enriquecido.

## 1. Configuração e Imports

In [ ]:
import os
import sys
import requests
import pandas as pd
from tqdm.notebook import tqdm

# ─── Configuração do CKAN ────────────────────────────────────────────────────
CKAN_BASE_URL = 'https://dadosabertos.web.stj.jus.br'

# Datasets de espelhos de acórdãos disponíveis na Base de Dados Abertos
DATASET_IDs = [
    ('S3', 'espelhos-de-acordaos-terceira-secao'),
    ('S2', 'espelhos-de-acordaos-segunda-secao'),
    ('S1', 'espelhos-de-acordaos-primeira-secao'),
    ('T1', 'espelhos-de-acordaos-primeira-turma'),
    ('T2', 'espelhos-de-acordaos-segunda-turma'),
    ('T3', 'espelhos-de-acordaos-terceira-turma'),
    ('T4', 'espelhos-de-acordaos-quarta-turma'),
    ('T5', 'espelhos-de-acordaos-quinta-turma'),
    ('T6', 'espelhos-de-acordaos-sexta-turma'),
    ('CE', 'espelhos-de-acordaos-corte-especial'),
]

# Campos de texto que serão extraídos do espelho (conforme ckan_extrair_espelhos.py)
CAMPOS_TEXTO = [
    'teseJuridica',
    'tema',
    'notas',
    'informacoesComplementares',
    'termosAuxiliares',
    'jurisprudenciaCitada',
    'referenciasLegislativas',
    'acordaosSimilares',
]

# Caminho do arquivo parquet base
PARQUET_BASE = 'espelhos_acordaos_artigo2026.parquet'
PARQUET_SAIDA = 'espelhos_acordaos_artigo2026_com_texto.parquet'

print(f'CKAN: {CKAN_BASE_URL}')
print(f'Datasets configurados: {len(DATASET_IDs)}')
print(f'Arquivo base: {PARQUET_BASE}')

## 2. Carregamento do Parquet Base

In [ ]:
# Carrega o parquet de espelhos base
df = pd.read_parquet(PARQUET_BASE)

print(f'Total de registros: {len(df)}')
print(f'Colunas: {df.columns.tolist()}')
print()
df.head()

In [ ]:
# Verifica coluna id_espelho
assert 'id_espelho' in df.columns, '❌ Coluna id_espelho não encontrada no parquet!'

ids_espelho = df['id_espelho'].dropna().unique().tolist()
print(f'✅ Total de id_espelho únicos: {len(ids_espelho)}')
print(f'Exemplos: {ids_espelho[:5]}')

## 3. Funções de Acesso ao CKAN

Seguindo o padrão de `ckan_extrair_espelhos.py`, as funções abaixo:
- Obtém os metadados de cada dataset para listar os recursos disponíveis.
- Constrói um índice `id_espelho → resource_id` para buscas eficientes.
- Consulta o texto de um espelho pelo `id_espelho` via `datastore_search`.

In [ ]:
def obter_metadados_dataset(dataset_id: str) -> dict:
    """Retorna os metadados de um dataset do CKAN, incluindo lista de recursos."""
    url = f'{CKAN_BASE_URL}/api/3/action/package_show'
    params = {'id': dataset_id}
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.json()['result']


def listar_resource_ids_json(dataset_id: str) -> list[dict]:
    """Retorna lista de {'resource_id', 'name'} dos recursos JSON do dataset."""
    metadados = obter_metadados_dataset(dataset_id)
    recursos_json = [
        {'resource_id': r['id'], 'name': r.get('name', ''), 'url': r.get('url', '')}
        for r in metadados.get('resources', [])
        if r.get('format', '').upper() == 'JSON'
    ]
    return recursos_json


# Teste rápido: lista recursos do primeiro dataset
orgao, dataset_id = DATASET_IDs[0]
print(f'Consultando recursos do dataset [{orgao}] "{dataset_id}"...')
try:
    recursos = listar_resource_ids_json(dataset_id)
    print(f'  → {len(recursos)} recursos JSON encontrados')
    for r in recursos[:3]:
        print(f'     {r["resource_id"]} | {r["name"]}')
except Exception as e:
    print(f'  ⚠️  Erro: {e}')

In [ ]:
def buscar_espelho_por_id(id_espelho: str, resource_id: str) -> dict | None:
    """
    Busca um registro de espelho pelo id_espelho no datastore do CKAN.
    Usa o endpoint datastore_search com filtro por 'id'.
    Retorna o registro completo ou None se não encontrado.
    """
    url = f'{CKAN_BASE_URL}/api/3/action/datastore_search'
    params = {
        'resource_id': resource_id,
        'filters': f'{{"id": "{id_espelho}"}}',
        'limit': 1,
    }
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        resultado = response.json()
        registros = resultado.get('result', {}).get('records', [])
        return registros[0] if registros else None
    except Exception:
        return None


def get_texto_espelho(registro: dict) -> str:
    """
    Extrai e consolida os campos de texto de um registro de espelho.
    Concatena teseJuridica, tema, notas e demais campos de texto relevantes.
    """
    partes = []
    for campo in CAMPOS_TEXTO:
        valor = registro.get(campo)
        if isinstance(valor, str) and valor.strip():
            partes.append(f'[{campo}]\n{valor.strip()}')
        elif isinstance(valor, list) and valor:
            partes.append(f'[{campo}]\n' + '\n'.join(str(v) for v in valor if v))
    return '\n\n'.join(partes)


print('✅ Funções de busca definidas.')

## 4. Construção do Índice de Recursos CKAN

Para evitar buscar em todos os recursos para cada `id_espelho`, constrói-se um índice de todos os `resource_id` disponíveis nos datasets de espelhos.

In [ ]:
# Coleta todos os resource_ids dos datasets de espelhos
todos_recursos = []  # lista de {'orgao', 'dataset_id', 'resource_id', 'name', 'url'}

print('Obtendo lista de recursos dos datasets de espelhos...')
for orgao, dataset_id in tqdm(DATASET_IDs, desc='Datasets'):
    try:
        recursos = listar_resource_ids_json(dataset_id)
        for r in recursos:
            todos_recursos.append({
                'orgao': orgao,
                'dataset_id': dataset_id,
                **r,
            })
        print(f'  [{orgao}] {dataset_id}: {len(recursos)} recursos JSON')
    except Exception as e:
        print(f'  ⚠️  [{orgao}] {dataset_id}: erro - {e}')

print(f'\nTotal de recursos CKAN coletados: {len(todos_recursos)}')

## 5. Carregamento do Texto por `id_espelho`

Para cada `id_espelho` no parquet, busca o registro correspondente no CKAN iterando pelos recursos até encontrá-lo.

In [ ]:
def buscar_texto_por_id_espelho(id_espelho: str, recursos: list[dict]) -> dict:
    """
    Itera pelos recursos CKAN até encontrar o registro com o id_espelho.
    Retorna dict com 'texto' (campos consolidados) e 'resource_id' encontrado.
    """
    if not id_espelho:
        return {'texto': '', 'resource_encontrado': None, 'encontrado': False}

    for recurso in recursos:
        registro = buscar_espelho_por_id(id_espelho, recurso['resource_id'])
        if registro:
            texto = get_texto_espelho(registro)
            return {
                'texto': texto,
                'resource_encontrado': recurso['resource_id'],
                'orgao_encontrado': recurso['orgao'],
                'encontrado': True,
            }

    return {'texto': '', 'resource_encontrado': None, 'orgao_encontrado': None, 'encontrado': False}


# Aplica a busca para todos os registros do parquet
resultados = []

print(f'Carregando textos de {len(df)} documentos via CKAN...')
for _, row in tqdm(df.iterrows(), total=len(df), desc='Buscando textos'):
    id_espelho = row.get('id_espelho')
    resultado = buscar_texto_por_id_espelho(str(id_espelho) if id_espelho else '', todos_recursos)
    resultados.append(resultado)

df_resultado = df.copy()
df_resultado['texto']             = [r['texto'] for r in resultados]
df_resultado['resource_ckan']     = [r['resource_encontrado'] for r in resultados]
df_resultado['orgao_ckan']        = [r['orgao_encontrado'] for r in resultados]

total_encontrados = sum(1 for r in resultados if r['encontrado'])
print(f'\n✅ Textos encontrados: {total_encontrados} / {len(df)}')
print(f'⚠️  Não encontrados:   {len(df) - total_encontrados} / {len(df)}')

## 6. Visualização dos Resultados

In [ ]:
# Resumo
print('=== RESUMO DO RESULTADO ===')
print(f'Total de registros: {len(df_resultado)}')
print(f'Com texto:          {df_resultado["texto"].str.len().gt(0).sum()}')
print(f'Sem texto:          {df_resultado["texto"].str.len().eq(0).sum()}')
print()

# Exibe os primeiros registros com texto
com_texto = df_resultado[df_resultado['texto'].str.len() > 0]
if not com_texto.empty:
    row = com_texto.iloc[0]
    print(f'--- Exemplo: id_espelho = {row["id_espelho"]} [{row.get("orgao_ckan","")}] ---')
    print(row['texto'][:2000])
    print('...')
else:
    print('Nenhum texto encontrado ainda.')

## 7. Salvamento do Resultado

In [ ]:
df_resultado.to_parquet(PARQUET_SAIDA, index=False)
print(f'✅ Arquivo salvo: {PARQUET_SAIDA}')
print(f'   Registros: {len(df_resultado)}')
print(f'   Colunas:   {df_resultado.columns.tolist()}')